# Diffusion-based Image Super-Resolution — Comparative Study
**Research Question:** Do diffusion-based models outperform GAN-based models in
perceptual quality of super-resolved images, and what is the trade-off with
inference speed and fidelity metrics?


## Section 0 — Pip Installs

In [1]:
!pip install -q lpips pytorch-fid scikit-image einops gdown


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.8 MB/s eta 0:00:00


## Section 1 — Imports & GPU Assert

In [2]:
import os, sys, math, random, time, json, csv, glob, copy
from pathlib import Path
from functools import partial
from collections import OrderedDict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.models import vgg19, VGG19_Weights

from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import lpips
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim

assert torch.cuda.is_available(), (
    "!! No GPU detected! Enable GPU: Runtime → Change runtime type → GPU."
)
DEVICE = torch.device("cuda")
print(f">GPU: {torch.cuda.get_device_name(0)}")
print(f">VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

>GPU: Tesla T4
>VRAM: 15.6 GB


## Section 2 — Seed & Reproducibility

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f">All random seeds fixed to {SEED}")

>All random seeds fixed to 42


#### 

# copy checkpoints from input to output


In [4]:
import shutil, os, torch
from collections import OrderedDict

INPUT_BASE = "/kaggle/input/datasets/shaharyarrizwan/genai-proj-output-before-results-ablation"
INPUT_CKPT = os.path.join(INPUT_BASE, "checkpoints")
INPUT_RESULTS = os.path.join(INPUT_BASE, "results")
WORK_CKPT     = "/kaggle/working/checkpoints"
WORK_RESULTS  = "/kaggle/working/results"

# ── Copy checkpoints ──
def strip_module_prefix(sd):
    if any(k.startswith("module.") for k in sd.keys()):
        return OrderedDict((k.replace("module.", "", 1), v) for k, v in sd.items())
    return sd

for model in ["ddpm", "esrgan"]:
    src_dir = os.path.join(INPUT_CKPT, model)
    dst_dir = os.path.join(WORK_CKPT, model)
    os.makedirs(dst_dir, exist_ok=True)
    for fname in os.listdir(src_dir):
        src_f = os.path.join(src_dir, fname)
        dst_f = os.path.join(dst_dir, fname)
        ckpt = torch.load(src_f, map_location="cpu", weights_only=False)
        if isinstance(ckpt, dict) and "model_state" in ckpt:
            ckpt["model_state"] = strip_module_prefix(ckpt["model_state"])
        elif isinstance(ckpt, dict):
            ckpt = strip_module_prefix(ckpt)
        torch.save(ckpt, dst_f)
        print(f">Copied+fixed {model}/{fname} ({os.path.getsize(dst_f)/1e6:.0f} MB)")

    # Create best_model.pt from ema_final/epoch_latest if missing
    best = os.path.join(WORK_CKPT, model, "best_model.pt")
    if not os.path.exists(best):
        fallback = os.path.join(WORK_CKPT, model,
                                "ema_final.pt" if model == "ddpm" else "epoch_latest.pt")
        if os.path.exists(fallback):
            shutil.copy2(fallback, best)
            print(f">Created best_model.pt from {os.path.basename(fallback)} for {model}")

print("\n>Done")
for model in ["ddpm", "esrgan"]:
    print(f"  {model}/:", os.listdir(os.path.join(WORK_CKPT, model)))

# ── Copy previous results (for eval/FID resume) ──
if os.path.exists(INPUT_RESULTS):
    shutil.copytree(INPUT_RESULTS, WORK_RESULTS, dirs_exist_ok=True)
    print(f"\n>Copied previous results from input:")
    for item in sorted(os.listdir(WORK_RESULTS)):
        item_path = os.path.join(WORK_RESULTS, item)
        if os.path.isdir(item_path):
            print(f"  {item}/: {len(os.listdir(item_path))} files")
        else:
            print(f"  {item}")
else:
    print("\n>No previous results found in input, starting fresh")

>Copied+fixed ddpm/ema_final.pt (230 MB)
>Copied+fixed ddpm/epoch_latest.pt (919 MB)
>Copied+fixed ddpm/best_model.pt (230 MB)
>Copied+fixed esrgan/epoch_latest.pt (290 MB)
>Copied+fixed esrgan/best_model.pt (68 MB)

>Done
  ddpm/: ['ema_final.pt', 'best_model.pt', 'epoch_latest.pt']
  esrgan/: ['best_model.pt', 'epoch_latest.pt']

>Copied previous results from input:
  fid_DDPM/: 35 files
  fid_hr/: 35 files
  metrics_summary.csv
  metrics_table.tex


## Section 3 — CONFIG Dict
Every hyperparameter lives here. **No magic numbers anywhere else.**

#### 

In [5]:
CONFIG = {
    # ── Paths (adjust dataset slugs to match your Kaggle inputs) ──
        "div2k_hr_dir":     "/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR",
        "set5_dir":         "/kaggle/input/datasets/aagamjnn/datasets-set5-set14-bsd100-urban100-manga109/benchmark/Set5/HR",
        "set14_dir":        "/kaggle/input/datasets/aagamjnn/datasets-set5-set14-bsd100-urban100-manga109/benchmark/Set14/HR",
        "bsd100_dir":       "/kaggle/input/datasets/aagamjnn/datasets-set5-set14-bsd100-urban100-manga109/benchmark/B100/HR",
        "esrgan_pretrained_path": "/kaggle/input/datasets/ahmedalizahid/esrgan-pretrained/RRDB_PSNR_x4.pth",
        "checkpoint_dir":   "/kaggle/working/checkpoints",
        "visual_dir":       "/kaggle/working/visuals",
        "results_dir":      "/kaggle/working/results",

    # ── Data ──
    "hr_patch_size": 128,
    "scale_factor":  4,
    "batch_size":    16,
    "num_workers":   2,

    # ── Diffusion (shared) ──
    "diffusion_steps": 1000,
    "beta_start":      1e-4,
    "beta_end":        0.02,
    "noise_schedule":  "linear",       # "linear" or "cosine"

    # ── DDPM / SR3 ──
    "ddpm_lr":        2e-4,
    "ddpm_epochs":    100,
    "ddpm_ema_decay": 0.999,
    "ddpm_perc_weight": 0.01,          # weight for perceptual loss (0 = disable)

    # ── DDIM ──
    "ddim_sampling_steps": 50,
    "ddim_eta":            0.0,        # 0 = deterministic

    # ── ESRGAN ──
    "esrgan_lr_g":           1e-4,
    "esrgan_lr_d":           1e-4,
    "esrgan_epochs_psnr":    100,
    "esrgan_epochs_gan":     100,
    "esrgan_num_rrdb":       23,       # match pretrained weights
    "esrgan_residual_scaling": 0.2,
    "loss_weight_l1":          1.0,
    "loss_weight_perceptual":  1.0,
    "loss_weight_adversarial": 0.005,

    # ── U-Net ──
    "unet_base_ch":     64,
    "unet_ch_mults":    (1, 2, 4, 8),
    "unet_num_res":     2,
    "unet_attn_res":    (16,),
    "unet_dropout":     0.0,

    # ── Evaluation ──
    "eval_border_crop": 4,
    "eval_y_channel":   True,

    # ── Ablation ──
    "ablation_steps":     [100, 200, 300],
    "ablation_schedules": ["linear", "cosine"],
    "ablation_scales":    [2, 4, 8],

    # ── Misc ──
    "seed": SEED,
    "amp":  True,
    "checkpoint_interval": 1,
}



# Create output directories
for d in [CONFIG["checkpoint_dir"], CONFIG["visual_dir"], CONFIG["results_dir"]]:
    os.makedirs(d, exist_ok=True)
for m in ["ddpm", "esrgan"]:
    os.makedirs(os.path.join(CONFIG["checkpoint_dir"], m), exist_ok=True)
print(">CONFIG loaded, output directories created")


>CONFIG loaded, output directories created


In [6]:
import os
for p in ["/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR",
          "/kaggle/input/datasets/aagamjnn/datasets-set5-set14-bsd100-urban100-manga109/benchmark/Set5/HR",
          "/kaggle/input/datasets/aagamjnn/datasets-set5-set14-bsd100-urban100-manga109/benchmark/B100/HR"]:
    files = os.listdir(p) if os.path.exists(p) else "NOT FOUND"
    print(f"{p}: {len(files) if isinstance(files, list) else files} items")



/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR: 800 items
/kaggle/input/datasets/aagamjnn/datasets-set5-set14-bsd100-urban100-manga109/benchmark/Set5/HR: 5 items
/kaggle/input/datasets/aagamjnn/datasets-set5-set14-bsd100-urban100-manga109/benchmark/B100/HR: 100 items


## Section 4 — Data Pipeline
`SRDataset` for training (random crops + augmentation) and `SRTestDataset` for
evaluation (full images, no augmentation).

In [7]:
class SRDataset(Dataset):
    """Training dataset: loads HR images, crops patches, generates LR via bicubic."""

    def __init__(self, hr_dir, patch_size=128, scale_factor=4, augment=True):
        """
        Args:
            hr_dir: path to folder of HR images
            patch_size: HR crop size (LR will be patch_size // scale_factor)
            scale_factor: downsampling factor
            augment: random flip / rotation
        """
        self.hr_paths = sorted(
            glob.glob(os.path.join(hr_dir, "*"))
        )
        self.hr_paths = [p for p in self.hr_paths
                         if p.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))]
        assert len(self.hr_paths) > 0, f"No images found in {hr_dir}"
        self.patch_size = patch_size
        self.scale = scale_factor
        self.augment = augment

    def __len__(self):
        return len(self.hr_paths)

    def __getitem__(self, idx):
        """Returns dict with keys: hr, lr, bicubic  (all tensors in [-1,1])."""
        img = Image.open(self.hr_paths[idx]).convert("RGB")
        # Random crop
        w, h = img.size
        ps = self.patch_size
        if w < ps or h < ps:
            img = TF.resize(img, max(ps, h), max(ps, w))
            w, h = img.size
        top  = random.randint(0, h - ps)
        left = random.randint(0, w - ps)
        hr = TF.crop(img, top, left, ps, ps)

        # Augmentation
        if self.augment:
            if random.random() < 0.5:
                hr = TF.hflip(hr)
            rot = random.choice([0, 90, 180, 270])
            if rot:
                hr = TF.rotate(hr, rot)

        hr = TF.to_tensor(hr)                           # [0,1]
        lr_size = ps // self.scale
        lr = F.interpolate(hr.unsqueeze(0), size=lr_size,
                           mode="bicubic", align_corners=False,
                           antialias=True).squeeze(0).clamp(0, 1)
        bic = F.interpolate(lr.unsqueeze(0), size=ps,
                            mode="bicubic", align_corners=False,
                            antialias=True).squeeze(0).clamp(0, 1)

        # Normalise to [-1, 1]
        hr  = hr  * 2.0 - 1.0
        lr  = lr  * 2.0 - 1.0
        bic = bic * 2.0 - 1.0
        return {"hr": hr, "lr": lr, "bicubic": bic}


class SRTestDataset(Dataset):
    """Test dataset: full images, no crop, no augment. Pads to multiple of scale."""

    def __init__(self, hr_dir, scale_factor=4):
        self.hr_paths = sorted(
            [p for p in glob.glob(os.path.join(hr_dir, "*"))
             if p.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))]
        )
        assert len(self.hr_paths) > 0, f"No images in {hr_dir}"
        self.scale = scale_factor

    def __len__(self):
        return len(self.hr_paths)

    def __getitem__(self, idx):
        img = Image.open(self.hr_paths[idx]).convert("RGB")
        hr = TF.to_tensor(img)
        _, h, w = hr.shape
        # Ensure divisible by scale * 8 (for U-Net downsampling)
        new_h = h - h % (self.scale * 8)
        new_w = w - w % (self.scale * 8)
        hr = hr[:, :new_h, :new_w]

        lr_h, lr_w = new_h // self.scale, new_w // self.scale
        lr = F.interpolate(hr.unsqueeze(0), size=(lr_h, lr_w),
                           mode="bicubic", align_corners=False,
                           antialias=True).squeeze(0).clamp(0, 1)
        bic = F.interpolate(lr.unsqueeze(0), size=(new_h, new_w),
                            mode="bicubic", align_corners=False,
                            antialias=True).squeeze(0).clamp(0, 1)
        hr  = hr  * 2.0 - 1.0
        lr  = lr  * 2.0 - 1.0
        bic = bic * 2.0 - 1.0
        return {"hr": hr, "lr": lr, "bicubic": bic,
                "filename": os.path.basename(self.hr_paths[idx])}


def build_dataloaders(config):
    """Build train and test dataloaders from CONFIG."""
    train_ds = SRDataset(
        config["div2k_hr_dir"],
        patch_size=config["hr_patch_size"],
        scale_factor=config["scale_factor"],
    )
    train_loader = DataLoader(
        train_ds, batch_size=config["batch_size"], shuffle=True,
        num_workers=config["num_workers"], pin_memory=True, drop_last=True, persistent_workers=True,
    )
    test_loaders = {}
    for name, path in [("Set5", config["set5_dir"]),
                       ("Set14", config["set14_dir"]),
                       ("BSD100", config["bsd100_dir"])]:
        ds = SRTestDataset(path, scale_factor=config["scale_factor"])
        test_loaders[name] = DataLoader(ds, batch_size=1, shuffle=False)
    return train_loader, test_loaders


# Quick validation: visualise a batch
def visualise_batch(loader, num=4):
    """Display LR / Bicubic / HR triplets from the first batch."""
    batch = next(iter(loader))
    fig, axes = plt.subplots(num, 3, figsize=(12, 4 * num))
    for i in range(min(num, batch["hr"].size(0))):
        for j, (key, title) in enumerate(
            [("lr", "LR"), ("bicubic", "Bicubic↑"), ("hr", "HR")]
        ):
            img = batch[key][i].permute(1, 2, 0).numpy() * 0.5 + 0.5
            axes[i, j].imshow(img.clip(0, 1))
            axes[i, j].set_title(title)
            axes[i, j].axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["visual_dir"], "batch_preview.png"), dpi=100)
    plt.close()
    print(">Batch preview saved to visuals/batch_preview.png")


print(">Data pipeline defined")

>Data pipeline defined


## Section 5 — Shared Utilities
Noise schedules, VGG perceptual loss, checkpoint save/load, EMA.

In [8]:
# ────────────────────── Noise Schedules ──────────────────────

def linear_beta_schedule(num_steps, beta_start=1e-4, beta_end=0.02):
    """Linear noise schedule (Ho et al. 2020)."""
    return torch.linspace(beta_start, beta_end, num_steps, dtype=torch.float64)


def cosine_beta_schedule(num_steps, s=0.008):
    """Cosine noise schedule (Nichol & Dhariwal 2021)."""
    t = torch.arange(num_steps + 1, dtype=torch.float64)
    f_t = torch.cos(((t / num_steps) + s) / (1 + s) * (math.pi / 2)) ** 2
    alphas_bar = f_t / f_t[0]
    betas = 1 - (alphas_bar[1:] / alphas_bar[:-1])
    return betas.clamp(0, 0.999)


def make_diffusion_schedule(schedule_type, num_steps, beta_start=1e-4, beta_end=0.02):
    """Build all pre-computed diffusion quantities.

    Returns a dict of tensors (float32, on CPU — move to GPU when needed).
    """
    if schedule_type == "linear":
        betas = linear_beta_schedule(num_steps, beta_start, beta_end)
    elif schedule_type == "cosine":
        betas = cosine_beta_schedule(num_steps)
    else:
        raise ValueError(f"Unknown schedule: {schedule_type}")

    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

    schedule = {
        "betas":                betas.float(),
        "alphas":               alphas.float(),
        "alphas_cumprod":       alphas_cumprod.float(),
        "alphas_cumprod_prev":  alphas_cumprod_prev.float(),
        "sqrt_alphas_cumprod":  alphas_cumprod.sqrt().float(),
        "sqrt_one_minus_ac":    (1.0 - alphas_cumprod).sqrt().float(),
        "sqrt_recip_ac":        (1.0 / alphas_cumprod).sqrt().float(),
        "sqrt_recip_ac_m1":     (1.0 / alphas_cumprod - 1).sqrt().float(),
        # Posterior q(x_{t-1} | x_t, x_0)
        "posterior_var":        (betas * (1.0 - alphas_cumprod_prev)
                                 / (1.0 - alphas_cumprod)).float(),
        "posterior_log_var":    torch.log(
            (betas * (1.0 - alphas_cumprod_prev)
             / (1.0 - alphas_cumprod)).clamp(min=1e-20)
        ).float(),
        "posterior_mean_c1":    (betas * alphas_cumprod_prev.sqrt()
                                 / (1.0 - alphas_cumprod)).float(),
        "posterior_mean_c2":    ((1.0 - alphas_cumprod_prev) * alphas.sqrt()
                                 / (1.0 - alphas_cumprod)).float(),
    }
    return schedule


# ────────────────────── VGG Perceptual Loss ──────────────────────

class VGGPerceptualLoss(nn.Module):
    """VGG19-based perceptual loss (features before activation)."""

    def __init__(self):
        super().__init__()
        vgg = vgg19(weights=VGG19_Weights.DEFAULT).features[:36].eval()
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg
        self.register_buffer(
            "mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer(
            "std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def _normalise(self, x):
        """Convert from [-1,1] to ImageNet normalisation."""
        x = x * 0.5 + 0.5  # -> [0,1]
        return (x - self.mean) / self.std

    def forward(self, pred, target):
        """Compute perceptual loss between pred and target (both in [-1,1])."""
        return F.l1_loss(self.vgg(self._normalise(pred)),
                         self.vgg(self._normalise(target)))


# ────────────────────── Checkpoint Utilities ──────────────────────

def save_checkpoint(model, optimizer, scheduler, epoch, best_metric,
                    loss_history, model_name, config, extra=None):
    """Save training checkpoint. Overwrites previous to save space."""
    ckpt_dir = os.path.join(config["checkpoint_dir"], model_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    state = {
        "model":        (model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()),
        "optimizer":    optimizer.state_dict(),
        "scheduler":    scheduler.state_dict() if scheduler else None,
        "epoch":        epoch,
        "best_metric":  best_metric,
        "loss_history": loss_history,
    }
    if extra:
        state.update(extra)
    torch.save(state, os.path.join(ckpt_dir, "epoch_latest.pt"))


def load_checkpoint(model, optimizer, scheduler, model_name, config, device,
                    extra_keys=None):
    """Load checkpoint if it exists. Returns (start_epoch, best_metric, history, extra).

    Also checks /kaggle/input/{model_name}-sr-checkpoints/ for cross-session
    persistence.
    """
    ckpt_dir = os.path.join(config["checkpoint_dir"], model_name)
    path = os.path.join(ckpt_dir, "epoch_latest.pt")

    # Fallback: search known input dataset locations (cross-session)
    import shutil
    search_paths = [
        # standalone dataset push
        f"/kaggle/input/{model_name}-sr-checkpoints/{model_name}/epoch_latest.pt",
        f"/kaggle/input/{model_name}-sr-checkpoints/epoch_latest.pt",
        # attached as notebook output dataset
        f"/kaggle/input/notebooks/ahmedalizahid/notebook10435a2a80/checkpoints/{model_name}/epoch_latest.pt",
        # generic notebook output fallback
        f"/kaggle/input/notebook10435a2a80/checkpoints/{model_name}/epoch_latest.pt",
    ]
    if not os.path.exists(path):
        for alt_path in search_paths:
            if os.path.exists(alt_path):
                os.makedirs(ckpt_dir, exist_ok=True)
                shutil.copy2(alt_path, path)
                print(f"📂 Copied checkpoint from: {alt_path}")
                break
        else:
            print(f"🔍 Searched paths for {model_name}:")
            for p in search_paths:
                print(f"   {p} — {'found' if os.path.exists(p) else 'missing'}")

    if not os.path.exists(path):
        print(f"🆕 No checkpoint for {model_name}, starting from scratch.")
        return 0, 0.0, [], {}

    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler and ckpt.get("scheduler"):
        scheduler.load_state_dict(ckpt["scheduler"])
    extra = {k: ckpt[k] for k in (extra_keys or []) if k in ckpt}
    print(f">Resumed {model_name} from epoch {ckpt['epoch']+1} "
          f"(best PSNR={ckpt['best_metric']:.2f} dB)")
    return ckpt["epoch"] + 1, ckpt["best_metric"], ckpt["loss_history"], extra


def save_best_model(model, metric, best_metric, model_name, config):
    """Save best model only if metric strictly improves."""
    if metric > best_metric:
        path = os.path.join(config["checkpoint_dir"], model_name, "best_model.pt")
        sd = model.state_dict()
        torch.save(sd, path)
        print(f"🏆 New best {model_name}: {metric:.2f} dB (was {best_metric:.2f})")
        return metric
    return best_metric


# ────────────────────── EMA ──────────────────────

class EMA:
    """Exponential Moving Average of model parameters."""

    def __init__(self, model, decay=0.999):
        self.decay = decay
        m = model
        self.shadow = {k: v.clone() for k, v in m.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        m = model
        for k, v in m.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v, alpha=1 - self.decay)

    def apply(self, model):
        m = model
        m.load_state_dict(self.shadow)

    def state_dict(self):
        return self.shadow

    def load_state_dict(self, sd):
        self.shadow = sd


# ────────────────────── Metrics Helpers ──────────────────────

def tensor_to_np(t):
    """Convert [-1,1] tensor (C,H,W) to numpy uint8 (H,W,C)."""
    img = t.detach().cpu().float().clamp(-1, 1) * 0.5 + 0.5
    return (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


def rgb_to_y(img_np):
    """Convert RGB uint8 to Y channel (ITU-R BT.601)."""
    return np.dot(img_np[..., :3], [65.481, 128.553, 24.966]) / 255.0 + 16.0


def calc_psnr_ssim(sr_np, hr_np, border=4, y_channel=True):
    """Compute PSNR and SSIM on cropped Y-channel (or RGB)."""
    if border > 0:
        sr_np = sr_np[border:-border, border:-border]
        hr_np = hr_np[border:-border, border:-border]
    if y_channel:
        sr_y = rgb_to_y(sr_np)
        hr_y = rgb_to_y(hr_np)
        psnr = compute_psnr(hr_y, sr_y, data_range=235.0 - 16.0)
        ssim = compute_ssim(hr_y, sr_y, data_range=235.0 - 16.0)
    else:
        psnr = compute_psnr(hr_np, sr_np, data_range=255)
        ssim = compute_ssim(hr_np, sr_np, data_range=255, channel_axis=2)
    return psnr, ssim


print(">Shared utilities defined")

>Shared utilities defined


## Section 6 — U-Net Architecture
Shared U-Net backbone for DDPM and DDIM. Input = concat(noisy\_HR, LR\_bicubic)
= 6 channels. Output = predicted noise ε (3 channels).

In [9]:
class SinusoidalPosEmb(nn.Module):
    """Sinusoidal positional embedding for diffusion timestep."""

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class ResBlock(nn.Module):
    """Residual block with FiLM conditioning from timestep embedding."""

    def __init__(self, in_ch, out_ch, time_dim, dropout=0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(32, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_ch * 2),  # scale + shift
        )
        self.norm2 = nn.GroupNorm(32, out_ch)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        """x: (B,C,H,W), t_emb: (B,D)."""
        h = self.conv1(F.silu(self.norm1(x)))
        # FiLM: scale and shift
        scale, shift = self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1).chunk(2, dim=1)
        h = self.norm2(h) * (1 + scale) + shift
        h = self.conv2(self.dropout(F.silu(h)))
        return h + self.skip(x)


class SelfAttention(nn.Module):
    """Multi-head self-attention with group-norm."""

    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(32, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
        self.num_heads = num_heads

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, self.num_heads, C // self.num_heads, H * W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.einsum("bhdn,bhdm->bhnm", q, k) * (C // self.num_heads) ** -0.5
        attn = attn.softmax(dim=-1)
        out = torch.einsum("bhnm,bhdm->bhdn", attn, v)
        out = out.reshape(B, C, H, W)
        return x + self.proj(out)


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return self.conv(x)


class UNet(nn.Module):
    """U-Net for SR3 / DDPM conditioned on LR image via concatenation.

    Architecture: Encoder → Bottleneck → Decoder with skip connections.
    Time conditioning via FiLM in every ResBlock.
    Self-attention at specified spatial resolutions.
    """

    def __init__(self, in_ch=6, out_ch=3, base_ch=64,
                 ch_mults=(1, 2, 4, 8), num_res=2,
                 attn_res=(16,), dropout=0.0):
        super().__init__()
        time_dim = base_ch * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(base_ch),
            nn.Linear(base_ch, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        # ── Encoder ──
        self.init_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)
        self.encoder = nn.ModuleList()
        self.encoder_ds = nn.ModuleList()
        channels = [base_ch]
        ch = base_ch
        current_res = 128  # assumed input spatial size for attn check

        for i, mult in enumerate(ch_mults):
            out = base_ch * mult
            blocks = nn.ModuleList()
            for _ in range(num_res):
                blocks.append(ResBlock(ch, out, time_dim, dropout))
                ch = out
                if current_res in attn_res:
                    blocks.append(SelfAttention(ch))
            self.encoder.append(blocks)
            channels.append(ch)
            if i < len(ch_mults) - 1:
                self.encoder_ds.append(Downsample(ch))
                current_res //= 2
            else:
                self.encoder_ds.append(nn.Identity())

        # ── Bottleneck ──
        self.bottleneck = nn.ModuleList([
            ResBlock(ch, ch, time_dim, dropout),
            SelfAttention(ch),
            ResBlock(ch, ch, time_dim, dropout),
        ])

        # ── Decoder ──
        self.decoder = nn.ModuleList()
        self.decoder_us = nn.ModuleList()
        for i, mult in reversed(list(enumerate(ch_mults))):
            out = base_ch * mult
            blocks = nn.ModuleList()
            for j in range(num_res + 1):
                skip_ch = channels.pop() if j == 0 else 0
                inp = ch + skip_ch if j == 0 else ch
                blocks.append(ResBlock(inp, out, time_dim, dropout))
                ch = out
                if current_res in attn_res:
                    blocks.append(SelfAttention(ch))
            self.decoder.append(blocks)
            if i > 0:
                self.decoder_us.append(Upsample(ch))
                current_res *= 2
            else:
                self.decoder_us.append(nn.Identity())

        self.final = nn.Sequential(
            nn.GroupNorm(32, ch),
            nn.SiLU(),
            nn.Conv2d(ch, out_ch, 3, padding=1),
        )

    def forward(self, x_noisy, lr_up, t):
        """
        Args:
            x_noisy: (B, 3, H, W) noisy HR image at timestep t
            lr_up:   (B, 3, H, W) bicubic-upscaled LR image
            t:       (B,) integer timesteps
        Returns:
            predicted noise ε: (B, 3, H, W)
        """
        t_emb = self.time_mlp(t)
        x = self.init_conv(torch.cat([x_noisy, lr_up], dim=1))

        # Encoder
        skips = [x]
        for blocks, ds in zip(self.encoder, self.encoder_ds):
            for block in blocks:
                if isinstance(block, ResBlock):
                    x = block(x, t_emb)
                else:
                    x = block(x)
            skips.append(x)
            x = ds(x)

        # Bottleneck
        for block in self.bottleneck:
            if isinstance(block, ResBlock):
                x = block(x, t_emb)
            else:
                x = block(x)

        # Decoder
        for blocks, us in zip(self.decoder, self.decoder_us):
            for i, block in enumerate(blocks):
                if isinstance(block, ResBlock):
                    if i == 0:
                        x = torch.cat([x, skips.pop()], dim=1)
                    x = block(x, t_emb)
                else:
                    x = block(x)
            x = us(x)

        return self.final(x)


print(">U-Net architecture defined")

>U-Net architecture defined


## Section 7 — DDPM / SR3 Training & Inference (Branch A)

Trains the U-Net to predict noise ε given (x\_t, LR, t).
Loss = L1(ε, ε\_pred) + λ · VGG\_perceptual(x0\_hat, x0).

In [10]:
def _extract(a, t, shape):
    """Gather values from tensor a at indices t and reshape for broadcasting."""
    out = a.gather(0, t)
    return out.view(-1, *((1,) * (len(shape) - 1)))


def q_sample(x_0, t, schedule, noise=None):
    """Forward diffusion: add noise to x_0 at timestep t."""
    if noise is None:
        noise = torch.randn_like(x_0)
    sqrt_ac = _extract(schedule["sqrt_alphas_cumprod"].to(x_0.device), t, x_0.shape)
    sqrt_om = _extract(schedule["sqrt_one_minus_ac"].to(x_0.device), t, x_0.shape)
    return sqrt_ac * x_0 + sqrt_om * noise, noise


@torch.no_grad()
def p_sample_step(model, x_t, t, lr_up, schedule):
    """Single DDPM reverse step: x_t → x_{t-1}."""
    betas = schedule["betas"].to(x_t.device)
    sqrt_recip = _extract(schedule["sqrt_recip_ac"].to(x_t.device), t, x_t.shape)
    sqrt_rm1   = _extract(schedule["sqrt_recip_ac_m1"].to(x_t.device), t, x_t.shape)
    pm_c1 = _extract(schedule["posterior_mean_c1"].to(x_t.device), t, x_t.shape)
    pm_c2 = _extract(schedule["posterior_mean_c2"].to(x_t.device), t, x_t.shape)
    post_var = _extract(schedule["posterior_var"].to(x_t.device), t, x_t.shape)

    eps_pred = model(x_t, lr_up, t)
    # Predict x_0
    x_0_hat = (sqrt_recip * x_t - sqrt_rm1 * eps_pred).clamp(-1, 1)
    # Posterior mean
    mean = pm_c1 * x_0_hat + pm_c2 * x_t
    noise = torch.randn_like(x_t) if t[0].item() > 0 else 0.0
    return mean + post_var.sqrt() * noise


@torch.no_grad()
def ddpm_sample(model, lr_up, schedule, num_steps=None):
    """Full DDPM reverse chain: pure noise → SR image.

    Args:
        model: trained U-Net
        lr_up: (B, 3, H, W) bicubic-upscaled LR in [-1,1]
        schedule: diffusion schedule dict
        num_steps: number of steps (default: len of schedule)
    Returns:
        SR image tensor (B, 3, H, W) in [-1,1]
    """
    B, C, H, W = lr_up.shape
    device = lr_up.device
    T = num_steps or len(schedule["betas"])
    x = torch.randn(B, C, H, W, device=device)
    for i in reversed(range(T)):
        t = torch.full((B,), i, device=device, dtype=torch.long)
        x = p_sample_step(model, x, t, lr_up, schedule)
    return x.clamp(-1, 1)


@torch.no_grad()
def quick_val_psnr(model, schedule, test_loader, device, max_imgs=4,
                   ddim_steps=20):
    """Quick validation using fast DDIM (20 steps) instead of full DDPM.
    """
    model.eval()
    sampler = DDIMSampler(schedule, ddim_steps=ddim_steps, eta=0.0)
    psnrs = []
    for i, batch in enumerate(test_loader):
        if i >= max_imgs:
            break
        lr_up = batch["bicubic"].to(device)
        hr    = batch["hr"].to(device)
        sr = sampler.sample(model, lr_up)
        sr_np = tensor_to_np(sr[0])
        hr_np = tensor_to_np(hr[0])
        p, _ = calc_psnr_ssim(sr_np, hr_np,
                               border=CONFIG["eval_border_crop"],
                               y_channel=CONFIG["eval_y_channel"])
        psnrs.append(p)
    return float(np.mean(psnrs)) if psnrs else 0.0


def train_ddpm(config):
    """Full DDPM / SR3 training loop with AMP, EMA, checkpointing.

    Automatically resumes from checkpoint if one exists.
    """
    print("=" * 60)
    print("  🔵 DDPM / SR3 TRAINING")
    print("=" * 60)

    # Data
    train_loader, test_loaders = build_dataloaders(config)
    visualise_batch(train_loader)
    val_loader = test_loaders["Set5"]

    # Model & schedule
    schedule = make_diffusion_schedule(
        config["noise_schedule"], config["diffusion_steps"],
        config["beta_start"], config["beta_end"],
    )
    model = UNet(
        in_ch=6, out_ch=3, base_ch=config["unet_base_ch"],
        ch_mults=config["unet_ch_mults"], num_res=config["unet_num_res"],
        attn_res=config["unet_attn_res"], dropout=config["unet_dropout"],
    ).to(DEVICE)
    print(f"U-Net params: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=config["ddpm_lr"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["ddpm_epochs"])
    scaler = GradScaler("cuda", enabled=config["amp"])

    # Perceptual loss (optional)
    vgg_loss = None
    if config["ddpm_perc_weight"] > 0:
        vgg_loss = VGGPerceptualLoss().to(DEVICE)

    # Resume
    start_epoch, best_psnr, loss_hist, extra = load_checkpoint(
        model, optimizer, scheduler, "ddpm", config, DEVICE,
        extra_keys=["ema_state"])
    ema = EMA(model, config["ddpm_ema_decay"])
    if "ema_state" in extra:
        ema.load_state_dict(extra["ema_state"])

    # Training loop
    T = config["diffusion_steps"]
    for epoch in range(start_epoch, config["ddpm_epochs"]):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['ddpm_epochs']}")
        for batch in pbar:
            hr    = batch["hr"].to(DEVICE)
            lr_up = batch["bicubic"].to(DEVICE)

            t = torch.randint(0, T, (hr.size(0),), device=DEVICE)
            noise = torch.randn_like(hr)
            x_t, _ = q_sample(hr, t, schedule, noise)

            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=config["amp"]):
                eps_pred = model(x_t, lr_up, t)
                loss = F.l1_loss(eps_pred, noise)
                # Optional perceptual loss on estimated x_0
                if vgg_loss is not None:
                    sqrt_ac = _extract(
                        schedule["sqrt_alphas_cumprod"].to(DEVICE), t, hr.shape)
                    sqrt_om = _extract(
                        schedule["sqrt_one_minus_ac"].to(DEVICE), t, hr.shape)
                    x0_hat = ((x_t - sqrt_om * eps_pred) / sqrt_ac).clamp(-1, 1)
                    loss = loss + config["ddpm_perc_weight"] * vgg_loss(x0_hat, hr).mean()

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            ema.update(model)

            epoch_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        scheduler.step()
        avg_loss = epoch_loss / len(train_loader)
        loss_hist.append(avg_loss)

        # Validation (using EMA weights)
        orig_sd = copy.deepcopy(model.state_dict())
        ema.apply(model)
        val_psnr = quick_val_psnr(model, schedule, val_loader, DEVICE)
        model.load_state_dict(orig_sd)

        print(f"  Epoch {epoch+1}: loss={avg_loss:.4f}  val_PSNR={val_psnr:.2f} dB")

        # Checkpoint
        best_psnr = save_best_model(model, val_psnr, best_psnr, "ddpm", config)
        save_checkpoint(model, optimizer, scheduler, epoch, best_psnr,
                        loss_hist, "ddpm", config,
                        extra={"ema_state": ema.state_dict()})

    # Save final EMA weights
    ema.apply(model)
    torch.save(model.state_dict(),
               os.path.join(config["checkpoint_dir"], "ddpm", "ema_final.pt"))
    print(">DDPM training complete")
    return model, schedule


print(">DDPM training functions defined")

>DDPM training functions defined


## Section 8 — DDIM Sampler (Branch B)
DDIM reuses the DDPM-trained model but uses a **deterministic** sampling
formula with far fewer steps (e.g., 50 vs 1000). No separate training needed.

In [11]:
class DDIMSampler:
    """DDIM deterministic sampler for accelerated inference.

    Given a DDPM-trained noise-prediction model, performs reverse diffusion
    using the DDIM update rule with a configurable number of steps and eta.
    """

    def __init__(self, schedule, ddim_steps=50, eta=0.0):
        """
        Args:
            schedule: diffusion schedule dict from make_diffusion_schedule
            ddim_steps: number of inference steps (≪ training T)
            eta: stochasticity (0 = fully deterministic DDIM)
        """
        self.ddim_steps = ddim_steps
        self.eta = eta
        T = len(schedule["alphas_cumprod"])
        # Sub-sequence of timesteps
        self.timesteps = torch.linspace(0, T - 1, ddim_steps + 1).long()
        self.alphas_cumprod = schedule["alphas_cumprod"]

    @torch.no_grad()
    def sample(self, model, lr_up):
        """Generate SR images using DDIM reverse process.

        Args:
            model: trained U-Net (noise predictor)
            lr_up: (B, 3, H, W) bicubic-upscaled LR in [-1,1]
        Returns:
            SR image (B, 3, H, W) in [-1,1]
        """
        device = lr_up.device
        B, C, H, W = lr_up.shape
        x = torch.randn(B, C, H, W, device=device)
        ac = self.alphas_cumprod.to(device)

        timesteps = self.timesteps.flip(0)  # reverse order: T → 0
        for i in range(len(timesteps) - 1):
            t_cur  = timesteps[i]
            t_prev = timesteps[i + 1]
            t_batch = torch.full((B,), t_cur, device=device, dtype=torch.long)

            eps_pred = model(x, lr_up, t_batch)

            a_t    = ac[t_cur]
            a_prev = ac[t_prev]

            # Predict x_0
            x0_hat = (x - (1 - a_t).sqrt() * eps_pred) / a_t.sqrt()
            x0_hat = x0_hat.clamp(-1, 1)

            # Compute sigma
            sigma = self.eta * (
                (1 - a_prev) / (1 - a_t) * (1 - a_t / a_prev)
            ).sqrt()

            # Direction pointing to x_t
            dir_xt = (1 - a_prev - sigma**2).sqrt() * eps_pred

            # DDIM update
            noise = torch.randn_like(x) if sigma > 0 else 0.0
            x = a_prev.sqrt() * x0_hat + dir_xt + sigma * noise

        return x.clamp(-1, 1)


def compare_inference_speed(model, schedule, ddim_sampler, lr_up, num_runs=3):
    """Time DDPM vs DDIM inference and report speedup.

    Args:
        model: trained U-Net
        schedule: DDPM schedule dict
        ddim_sampler: DDIMSampler instance
        lr_up: (1, 3, H, W) single test image
        num_runs: number of timing runs for averaging
    Returns:
        dict with timing results
    """
    model.eval()
    device = lr_up.device
    results = {}

    # DDPM
    torch.cuda.synchronize()
    ddpm_times = []
    for _ in range(num_runs):
        start = time.time()
        _ = ddpm_sample(model, lr_up, schedule)
        torch.cuda.synchronize()
        ddpm_times.append(time.time() - start)
    results["ddpm_time"] = np.mean(ddpm_times)

    # DDIM
    torch.cuda.synchronize()
    ddim_times = []
    for _ in range(num_runs):
        start = time.time()
        _ = ddim_sampler.sample(model, lr_up)
        torch.cuda.synchronize()
        ddim_times.append(time.time() - start)
    results["ddim_time"] = np.mean(ddim_times)
    results["speedup"] = results["ddpm_time"] / results["ddim_time"]

    print(f"⏱  DDPM ({len(schedule['betas'])} steps): {results['ddpm_time']:.2f}s")
    print(f"⏱  DDIM ({ddim_sampler.ddim_steps} steps): {results['ddim_time']:.2f}s")
    print(f"⚡ Speedup: {results['speedup']:.1f}×")
    return results


print(">DDIM sampler defined")

>DDIM sampler defined


## Section 9 — ESRGAN Architecture (Branch C)

Generator = RRDBNet (Residual-in-Residual Dense Blocks, no BatchNorm).
Discriminator = VGG-style with Relativistic Average GAN (RaGAN) loss.

In [12]:
class DenseBlock(nn.Module):
    """Dense block with 5 conv layers, growth rate 32."""

    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.conv1 = nn.Conv2d(nf,          gc, 3, 1, 1)
        self.conv2 = nn.Conv2d(nf + gc,     gc, 3, 1, 1)
        self.conv3 = nn.Conv2d(nf + 2 * gc, gc, 3, 1, 1)
        self.conv4 = nn.Conv2d(nf + 3 * gc, gc, 3, 1, 1)
        self.conv5 = nn.Conv2d(nf + 4 * gc, nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat([x, x1], 1)))
        x3 = self.lrelu(self.conv3(torch.cat([x, x1, x2], 1)))
        x4 = self.lrelu(self.conv4(torch.cat([x, x1, x2, x3], 1)))
        x5 = self.conv5(torch.cat([x, x1, x2, x3, x4], 1))
        return x5 * 0.2 + x  # residual scaling


class RRDB(nn.Module):
    """Residual-in-Residual Dense Block (3 dense blocks)."""

    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.rdb1 = DenseBlock(nf, gc)
        self.rdb2 = DenseBlock(nf, gc)
        self.rdb3 = DenseBlock(nf, gc)

    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out * 0.2 + x


class RRDBNet(nn.Module):
    """ESRGAN Generator: RRDB-based network with PixelShuffle upsampling.

    Follows the original ESRGAN paper: no BatchNorm, residual scaling 0.2.
    """

    def __init__(self, in_nc=3, out_nc=3, nf=64, nb=23, gc=32, scale=4):
        super().__init__()
        self.scale = scale
        self.conv_first = nn.Conv2d(in_nc, nf, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(nf, gc) for _ in range(nb)])
        self.conv_body = nn.Conv2d(nf, nf, 3, 1, 1)

        # Upsampling
        upsample_blocks = []
        num_up = int(math.log2(scale))
        for _ in range(num_up):
            upsample_blocks += [
                nn.Conv2d(nf, nf * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.LeakyReLU(0.2, inplace=True),
            ]
        self.upsampler = nn.Sequential(*upsample_blocks)
        self.conv_hr = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_last = nn.Conv2d(nf, out_nc, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        """x: LR image in [-1,1]. Returns SR in [-1,1]."""
        feat = self.conv_first(x)
        body = self.conv_body(self.body(feat))
        feat = feat + body
        feat = self.upsampler(feat)
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return out


class VGGDiscriminator(nn.Module):
    """VGG-style discriminator for ESRGAN (Relativistic Average GAN)."""

    def __init__(self, in_nc=3, nf=64):
        super().__init__()
        layers = [
            nn.Conv2d(in_nc, nf, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf, nf, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf, nf * 2, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf * 2, nf * 2, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf * 2, nf * 4, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf * 4, nf * 4, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf * 4, nf * 8, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf * 8, nf * 8, 4, 2, 1), nn.LeakyReLU(0.2, True),
        ]
        self.features = nn.Sequential(*layers)
        # Adaptive pooling → classifier
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(nf * 8, 100),
            nn.LeakyReLU(0.2, True),
            nn.Linear(100, 1),
        )

    def forward(self, x):
        feat = self.features(x)
        return self.classifier(feat)


def load_pretrained_esrgan(generator, path, device):
    """Load pretrained ESRGAN weights (handles key mismatches gracefully)."""
    if not os.path.exists(path):
        print(f"⚠️  Pretrained weights not found at {path}, training from scratch.")
        return False
    state = torch.load(path, map_location=device, weights_only=False)
    # Handle nested state dicts (some saves wrap in 'params' or 'params_ema')
    if "params" in state:
        state = state["params"]
    elif "params_ema" in state:
        state = state["params_ema"]
    try:
        generator.load_state_dict(state, strict=True)
        print(f">Loaded pretrained ESRGAN from {path}")
    except RuntimeError:
        generator.load_state_dict(state, strict=False)
        print(f"⚠️  Loaded pretrained ESRGAN (strict=False) from {path}")
    return True


print(">ESRGAN architecture defined")

>ESRGAN architecture defined


## Section 10 — ESRGAN Training

Two-phase training:
- Phase 1 (PSNR-oriented): L1 loss only
- Phase 2 (GAN): L1 + Perceptual + Adversarial (RaGAN)

In [13]:
def train_esrgan(config):
    """Full ESRGAN two-phase training with checkpoint/resume support.

    Phase 1: Train generator with L1 loss for PSNR-oriented initialization.
    Phase 2: Train generator + discriminator with L1 + perceptual + RaGAN loss.
    """
    print("=" * 60)
    print("  🟢 ESRGAN TRAINING")
    print("=" * 60)

    # Data
    train_loader, test_loaders = build_dataloaders(config)
    val_loader = test_loaders["Set5"]

    # Models
    gen = RRDBNet(
        in_nc=3, out_nc=3, nf=64,
        nb=config["esrgan_num_rrdb"],
        scale=config["scale_factor"]
    ).to(DEVICE)
    disc = VGGDiscriminator().to(DEVICE)

    # Try to load pretrained generator weights
    pretrained_loaded = load_pretrained_esrgan(
        gen, config["esrgan_pretrained_path"], DEVICE)

    print(f"Generator params: {sum(p.numel() for p in gen.parameters()):,}")
    print(f"Discriminator params: {sum(p.numel() for p in disc.parameters()):,}")

    # Losses
    vgg_loss = VGGPerceptualLoss().to(DEVICE)
    lpips_fn = lpips.LPIPS(net="alex").to(DEVICE)

    # Optimizers
    opt_g = torch.optim.Adam(gen.parameters(), lr=config["esrgan_lr_g"],
                             betas=(0.9, 0.999))
    opt_d = torch.optim.Adam(disc.parameters(), lr=config["esrgan_lr_d"],
                             betas=(0.9, 0.999))
    sched_g = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_g, T_max=config["esrgan_epochs_psnr"] + config["esrgan_epochs_gan"])
    sched_d = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_d, T_max=config["esrgan_epochs_gan"])
    scaler = GradScaler("cuda", enabled=config["amp"])

    # Resume
    start_epoch, best_psnr, loss_hist, extra = load_checkpoint(
        gen, opt_g, sched_g, "esrgan", config, DEVICE,
        extra_keys=["disc_state", "opt_d_state", "phase"])
    if "disc_state" in extra:
        disc.load_state_dict(extra["disc_state"])
    if "opt_d_state" in extra:
        opt_d.load_state_dict(extra["opt_d_state"])
    current_phase = extra.get("phase", 1)

    total_psnr_epochs = config["esrgan_epochs_psnr"]
    total_gan_epochs  = config["esrgan_epochs_gan"]
    total_epochs = total_psnr_epochs + total_gan_epochs

    for epoch in range(start_epoch, total_epochs):
        phase = 1 if epoch < total_psnr_epochs else 2
        if pretrained_loaded and phase == 1:
            # Skip Phase 1 if pretrained weights loaded
            if epoch == start_epoch:
                print("⏩ Skipping Phase 1 (pretrained weights loaded)")
            continue

        gen.train()
        disc.train() if phase == 2 else None
        epoch_loss_g = 0.0
        pbar = tqdm(train_loader,
                    desc=f"Epoch {epoch+1}/{total_epochs} [Phase {phase}]")

        for batch in pbar:
            hr = batch["hr"].to(DEVICE)
            lr = batch["lr"].to(DEVICE)

            # Generator forward
            opt_g.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=config["amp"]):
                sr = gen(lr)
                loss_g = config["loss_weight_l1"] * F.l1_loss(sr, hr)

                if phase == 2:
                    # Perceptual loss
                    loss_g += config["loss_weight_perceptual"] * vgg_loss(sr, hr).mean()
                    # Relativistic average GAN loss
                    fake_pred = disc(sr)
                    real_pred = disc(hr).detach()
                    loss_g += config["loss_weight_adversarial"] * (
                        F.binary_cross_entropy_with_logits(
                            fake_pred - real_pred.mean(), torch.ones_like(fake_pred))
                    )

            scaler.scale(loss_g).backward()
            scaler.step(opt_g)
            scaler.update()

            # Discriminator (Phase 2 only)
            if phase == 2:
                opt_d.zero_grad(set_to_none=True)
                with autocast("cuda", enabled=config["amp"]):
                    real_pred = disc(hr)
                    fake_pred = disc(sr.detach())
                    loss_d = (
                        F.binary_cross_entropy_with_logits(
                            real_pred - fake_pred.mean(), torch.ones_like(real_pred))
                        + F.binary_cross_entropy_with_logits(
                            fake_pred - real_pred.mean(), torch.zeros_like(fake_pred))
                    ) * 0.5
                scaler.scale(loss_d).backward()
                scaler.step(opt_d)
                scaler.update()

            epoch_loss_g += loss_g.item()
            pbar.set_postfix(loss_g=f"{loss_g.item():.4f}")

        sched_g.step()
        if phase == 2:
            sched_d.step()

        avg_loss = epoch_loss_g / len(train_loader)
        loss_hist.append(avg_loss)

        # Validation PSNR
        gen.eval()
        val_psnrs = []
        with torch.no_grad():
            for i, batch in enumerate(val_loader):
                if i >= 5:
                    break
                lr = batch["lr"].to(DEVICE)
                hr = batch["hr"].to(DEVICE)
                sr = gen(lr)
                sr_np = tensor_to_np(sr[0])
                hr_np = tensor_to_np(hr[0])
                p, _ = calc_psnr_ssim(sr_np, hr_np,
                                       border=config["eval_border_crop"],
                                       y_channel=config["eval_y_channel"])
                val_psnrs.append(p)
        val_psnr = float(np.mean(val_psnrs)) if val_psnrs else 0.0
        print(f"  Epoch {epoch+1}: loss_g={avg_loss:.4f}  val_PSNR={val_psnr:.2f} dB")

        # Checkpoint
        best_psnr = save_best_model(gen, val_psnr, best_psnr, "esrgan", config)
        save_checkpoint(gen, opt_g, sched_g, epoch, best_psnr, loss_hist,
                        "esrgan", config,
                        extra={"disc_state": disc.state_dict(),
                               "opt_d_state": opt_d.state_dict(),
                               "phase": phase})

    print(">ESRGAN training complete")
    return gen


print(">ESRGAN training functions defined")

>ESRGAN training functions defined


## Section 11 — Evaluation Script

Computes PSNR, SSIM, LPIPS on all test sets; FID on BSD100 only.

In [14]:
# @torch.no_grad()
# def evaluate_all_models(config):
#     """Evaluate DDPM, DDIM, and ESRGAN on Set5, Set14, BSD100.

#     Returns a list of result dicts and saves CSV + LaTeX table.
#     Loads best_model.pt for each model from the checkpoint directory.
#     """
#     print("=" * 60)
#     print("  📊 FULL EVALUATION")
#     print("=" * 60)

#     _, test_loaders = build_dataloaders(config)
#     lpips_fn = lpips.LPIPS(net="alex").to(DEVICE)

#     # ── Load models ──
#     schedule = make_diffusion_schedule(
#         config["noise_schedule"], config["diffusion_steps"],
#         config["beta_start"], config["beta_end"],
#     )
#     ddim_sampler = DDIMSampler(schedule, config["ddim_sampling_steps"],
#                                config["ddim_eta"])

#     # DDPM model
#     unet = UNet(
#         in_ch=6, out_ch=3, base_ch=config["unet_base_ch"],
#         ch_mults=config["unet_ch_mults"], num_res=config["unet_num_res"],
#         attn_res=config["unet_attn_res"], dropout=config["unet_dropout"],
#     ).to(DEVICE)
#     ddpm_path = os.path.join(config["checkpoint_dir"], "ddpm", "best_model.pt")
#     if os.path.exists(ddpm_path):
#         unet.load_state_dict(torch.load(ddpm_path, map_location=DEVICE,
#                                         weights_only=False))
#         print(">Loaded DDPM best model")
#     else:
#         print("⚠️  DDPM best_model.pt not found, skipping diffusion evaluation")
#         unet = None

#     # ESRGAN model
#     esrgan = RRDBNet(
#         in_nc=3, out_nc=3, nf=64, nb=config["esrgan_num_rrdb"],
#         scale=config["scale_factor"],
#     ).to(DEVICE)
#     esrgan_path = os.path.join(config["checkpoint_dir"], "esrgan", "best_model.pt")
#     if os.path.exists(esrgan_path):
#         esrgan.load_state_dict(torch.load(esrgan_path, map_location=DEVICE,
#                                           weights_only=False))
#         print(">Loaded ESRGAN best model")
#     else:
#         print("⚠️  ESRGAN best_model.pt not found, skipping ESRGAN evaluation")
#         esrgan = None

#     # ── Define inference functions ──
#     inference_fns = {}
#     if unet is not None:
#         unet.eval()
#         inference_fns["DDPM"] = lambda bic: ddpm_sample(unet, bic, schedule)
#         inference_fns["DDIM"] = lambda bic: ddim_sampler.sample(unet, bic)
#     if esrgan is not None:
#         esrgan.eval()
#         inference_fns["ESRGAN"] = lambda lr: esrgan(lr)

#     all_results = []

#     for ds_name, loader in test_loaders.items():
#         print(f"\n── Evaluating on {ds_name} ({len(loader)} images) ──")
#         for model_name, inf_fn in inference_fns.items():
#             psnrs, ssims, lpips_vals = [], [], []
#             times = []

#             for batch in tqdm(loader, desc=f"  {model_name}"):
#                 hr  = batch["hr"].to(DEVICE)
#                 lr  = batch["lr"].to(DEVICE)
#                 bic = batch["bicubic"].to(DEVICE)

#                 # Inference
#                 t0 = time.time()
#                 if model_name == "ESRGAN":
#                     sr = inf_fn(lr)
#                 else:
#                     sr = inf_fn(bic)
#                 torch.cuda.synchronize()
#                 times.append(time.time() - t0)

#                 sr_np = tensor_to_np(sr[0])
#                 hr_np = tensor_to_np(hr[0])

#                 p, s = calc_psnr_ssim(sr_np, hr_np,
#                                        border=config["eval_border_crop"],
#                                        y_channel=config["eval_y_channel"])
#                 psnrs.append(p)
#                 ssims.append(s)

#                 # LPIPS (expects [0,1])
#                 sr_01 = sr * 0.5 + 0.5
#                 hr_01 = hr * 0.5 + 0.5
#                 lp = lpips_fn(sr_01, hr_01).mean().item()
#                 lpips_vals.append(lp)

#             result = {
#                 "model":    model_name,
#                 "dataset":  ds_name,
#                 "scale":    config["scale_factor"],
#                 "psnr":     float(np.mean(psnrs)),
#                 "ssim":     float(np.mean(ssims)),
#                 "lpips":    float(np.mean(lpips_vals)),
#                 "time_avg": float(np.mean(times)),
#             }
#             all_results.append(result)
#             print(f"    {model_name}: PSNR={result['psnr']:.2f}  "
#                   f"SSIM={result['ssim']:.4f}  LPIPS={result['lpips']:.4f}  "
#                   f"Time={result['time_avg']:.2f}s")

#     # ── FID on BSD100 only ──
#     if unet is not None or esrgan is not None:
#         print("\n── Computing FID on BSD100 ──")
#         bsd_loader = test_loaders["BSD100"]
#         fid_hr_dir = os.path.join(config["results_dir"], "fid_hr")
#         os.makedirs(fid_hr_dir, exist_ok=True)

#         for model_name, inf_fn in inference_fns.items():
#             fid_sr_dir = os.path.join(config["results_dir"], f"fid_{model_name}")
#             os.makedirs(fid_sr_dir, exist_ok=True)
#             for i, batch in enumerate(bsd_loader):
#                 hr  = batch["hr"].to(DEVICE)
#                 lr  = batch["lr"].to(DEVICE)
#                 bic = batch["bicubic"].to(DEVICE)
#                 fname = batch.get("filename", [f"{i:04d}.png"])[0]

#                 sr = inf_fn(bic) if model_name != "ESRGAN" else inf_fn(lr)
#                 Image.fromarray(tensor_to_np(sr[0])).save(
#                     os.path.join(fid_sr_dir, fname))
#                 if model_name == list(inference_fns.keys())[0]:
#                     Image.fromarray(tensor_to_np(hr[0])).save(
#                         os.path.join(fid_hr_dir, fname))

#             # Compute FID using pytorch_fid
#             try:
#                 from pytorch_fid import fid_score
#                 fid = fid_score.calculate_fid_given_paths(
#                     [fid_hr_dir, fid_sr_dir], batch_size=50,
#                     device=DEVICE, dims=2048)
#                 # Attach FID to the BSD100 result
#                 for r in all_results:
#                     if r["model"] == model_name and r["dataset"] == "BSD100":
#                         r["fid"] = float(fid)
#                 print(f"  {model_name} FID: {fid:.2f}")
#             except Exception as e:
#                 print(f"  ⚠️  FID computation failed for {model_name}: {e}")

#     # ── Save results ──
#     _save_results_csv(all_results, config)
#     _save_results_latex(all_results, config)
#     return all_results


# def _save_results_csv(results, config):
#     """Save evaluation results as CSV."""
#     path = os.path.join(config["results_dir"], "metrics_summary.csv")
#     keys = ["model", "dataset", "scale", "psnr", "ssim", "lpips", "fid", "time_avg"]
#     with open(path, "w", newline="") as f:
#         w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
#         w.writeheader()
#         w.writerows(results)
#     print(f">CSV saved: {path}")


# def _save_results_latex(results, config):
#     """Save evaluation results as LaTeX table."""
#     path = os.path.join(config["results_dir"], "metrics_table.tex")
#     with open(path, "w") as f:
#         f.write("\\begin{table}[h]\n\\centering\n")
#         f.write("\\caption{Super-resolution results at $\\times$"
#         f"{config['scale_factor']}}}\n")
#         f.write("\\begin{tabular}{llcccc}\n\\toprule\n")
#         f.write("Model & Dataset & PSNR$\\uparrow$ & SSIM$\\uparrow$ "
#                 "& LPIPS$\\downarrow$ & FID$\\downarrow$ \\\\\n\\midrule\n")
#         for r in results:
#             fid_str = f"{r['fid']:.2f}" if "fid" in r else "—"
#             f.write(f"{r['model']} & {r['dataset']} & {r['psnr']:.2f} & "
#                     f"{r['ssim']:.4f} & {r['lpips']:.4f} & {fid_str} \\\\\n")
#         f.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")
#     print(f">LaTeX table saved: {path}")


# print(">Evaluation functions defined")

In [15]:
@torch.no_grad()
def evaluate_all_models(config):
    """Evaluate DDPM, DDIM, and ESRGAN on Set5, Set14, BSD100.

    Returns a list of result dicts and saves CSV + LaTeX table.
    Loads best_model.pt for each model from the checkpoint directory.
    Supports resuming from a previous partial run via metrics_summary.csv.
    """
    print("=" * 60)
    print("  📊 FULL EVALUATION")
    print("=" * 60)

    _, test_loaders = build_dataloaders(config)
    lpips_fn = lpips.LPIPS(net="alex").to(DEVICE)

    # ── Load models ──
    schedule = make_diffusion_schedule(
        config["noise_schedule"], config["diffusion_steps"],
        config["beta_start"], config["beta_end"],
    )
    ddim_sampler = DDIMSampler(schedule, config["ddim_sampling_steps"],
                               config["ddim_eta"])

    # DDPM model
    unet = UNet(
        in_ch=6, out_ch=3, base_ch=config["unet_base_ch"],
        ch_mults=config["unet_ch_mults"], num_res=config["unet_num_res"],
        attn_res=config["unet_attn_res"], dropout=config["unet_dropout"],
    ).to(DEVICE)
    ddpm_path = os.path.join(config["checkpoint_dir"], "ddpm", "best_model.pt")
    if os.path.exists(ddpm_path):
        unet.load_state_dict(torch.load(ddpm_path, map_location=DEVICE,
                                        weights_only=False))
        print(">Loaded DDPM best model")
    else:
        print("⚠️  DDPM best_model.pt not found, skipping diffusion evaluation")
        unet = None

    # ESRGAN model
    esrgan = RRDBNet(
        in_nc=3, out_nc=3, nf=64, nb=config["esrgan_num_rrdb"],
        scale=config["scale_factor"],
    ).to(DEVICE)
    esrgan_path = os.path.join(config["checkpoint_dir"], "esrgan", "best_model.pt")
    if os.path.exists(esrgan_path):
        esrgan.load_state_dict(torch.load(esrgan_path, map_location=DEVICE,
                                          weights_only=False))
        print(">Loaded ESRGAN best model")
    else:
        print("⚠️  ESRGAN best_model.pt not found, skipping ESRGAN evaluation")
        esrgan = None

    # ── Define inference functions ──
    inference_fns = {}
    if unet is not None:
        unet.eval()
        inference_fns["DDPM"] = lambda bic: ddpm_sample(unet, bic, schedule)
        inference_fns["DDIM"] = lambda bic: ddim_sampler.sample(unet, bic)
    if esrgan is not None:
        esrgan.eval()
        inference_fns["ESRGAN"] = lambda lr: esrgan(lr)

    # ── Resume: load already-completed results from CSV ──
    import csv as _csv_mod
    _partial_csv = os.path.join(config["results_dir"], "metrics_summary.csv")
    all_results = []
    _completed = set()      # (model_name, dataset_name) pairs already done
    _completed_fid = set()  # model_names whose FID is already computed
    if os.path.exists(_partial_csv):
        with open(_partial_csv) as _f:
            for _row in _csv_mod.DictReader(_f):
                for _k in ["psnr", "ssim", "lpips", "time_avg", "fid"]:
                    if _k in _row and _row[_k] not in ("", "None", None):
                        try: _row[_k] = float(_row[_k])
                        except: pass
                all_results.append(_row)
                _completed.add((_row["model"], _row["dataset"]))
                if _row["dataset"] == "BSD100" and "fid" in _row and isinstance(_row["fid"], float):
                    _completed_fid.add(_row["model"])
        print(f">Resuming eval — {len(_completed)} (model, dataset) pairs already done; "
              f"{len(_completed_fid)} FID(s) already computed")

    # ── Per-dataset / per-model metrics ──
    for ds_name, loader in test_loaders.items():
        print(f"\n── Evaluating on {ds_name} ({len(loader)} images) ──")
        for model_name, inf_fn in inference_fns.items():

            # ✅ SKIP if already completed
            if (model_name, ds_name) in _completed:
                print(f"  ⏩ Skipping {model_name}/{ds_name} (already in CSV)")
                continue

            psnrs, ssims, lpips_vals = [], [], []
            times = []

            for batch in tqdm(loader, desc=f"  {model_name}"):
                hr  = batch["hr"].to(DEVICE)
                lr  = batch["lr"].to(DEVICE)
                bic = batch["bicubic"].to(DEVICE)

                t0 = time.time()
                if model_name == "ESRGAN":
                    sr = inf_fn(lr)
                else:
                    sr = inf_fn(bic)
                torch.cuda.synchronize()
                times.append(time.time() - t0)

                sr_np = tensor_to_np(sr[0])
                hr_np = tensor_to_np(hr[0])

                p, s = calc_psnr_ssim(sr_np, hr_np,
                                       border=config["eval_border_crop"],
                                       y_channel=config["eval_y_channel"])
                psnrs.append(p)
                ssims.append(s)

                sr_01 = sr * 0.5 + 0.5
                hr_01 = hr * 0.5 + 0.5
                lp = lpips_fn(sr_01, hr_01).mean().item()
                lpips_vals.append(lp)

            result = {
                "model":    model_name,
                "dataset":  ds_name,
                "scale":    config["scale_factor"],
                "psnr":     float(np.mean(psnrs)),
                "ssim":     float(np.mean(ssims)),
                "lpips":    float(np.mean(lpips_vals)),
                "time_avg": float(np.mean(times)),
            }
            all_results.append(result)
            print(f"    {model_name}: PSNR={result['psnr']:.2f}  "
                  f"SSIM={result['ssim']:.4f}  LPIPS={result['lpips']:.4f}  "
                  f"Time={result['time_avg']:.2f}s")
            # Save after every model/dataset so progress survives a crash
            _save_results_csv(all_results, config)
            _save_results_latex(all_results, config)

    # ── FID on BSD100 only ──
    if unet is not None or esrgan is not None:
        print("\n── Computing FID on BSD100 ──")
        bsd_loader = test_loaders["BSD100"]
        fid_hr_dir = os.path.join(config["results_dir"], "fid_hr")
        os.makedirs(fid_hr_dir, exist_ok=True)

        for model_name, inf_fn in inference_fns.items():
            fid_sr_dir = os.path.join(config["results_dir"], f"fid_{model_name}")
            os.makedirs(fid_sr_dir, exist_ok=True)

            # ✅ Count existing SR images to decide whether to generate
            expected = len(bsd_loader.dataset)
            existing_count = len([f for f in os.listdir(fid_sr_dir) if f.endswith(".png")])
            existing_sr_images = existing_count >= expected

            if existing_sr_images:
                print(f"  ⏩ Skipping {model_name} FID image generation "
                      f"({existing_count}/{expected} images already exist)")
            else:
                if existing_count > 0:
                    print(f"  ⚠️  Resuming {model_name} FID image generation "
                          f"({existing_count}/{expected} already done, skipping those)...")
                else:
                    print(f"  Generating SR images for {model_name} FID...")

                for i, batch in enumerate(tqdm(bsd_loader, desc=f"  {model_name} FID images")):
                    fname = batch.get("filename", [f"{i:04d}.png"])[0]
                    out_path = os.path.join(fid_sr_dir, fname)

                    # ✅ Per-image skip — don't regenerate what's already saved
                    if os.path.exists(out_path):
                        continue

                    hr  = batch["hr"].to(DEVICE)
                    lr  = batch["lr"].to(DEVICE)
                    bic = batch["bicubic"].to(DEVICE)

                    sr = inf_fn(bic) if model_name != "ESRGAN" else inf_fn(lr)
                    Image.fromarray(tensor_to_np(sr[0])).save(out_path)

                    # Save HR reference images only once
                    hr_fname = os.path.join(fid_hr_dir, fname)
                    if not os.path.exists(hr_fname):
                        Image.fromarray(tensor_to_np(hr[0])).save(hr_fname)

            # ✅ SKIP FID score computation if already in results
            if model_name in _completed_fid:
                print(f"  ⏩ Skipping {model_name} FID score (already in CSV)")
                continue

            try:
                from pytorch_fid import fid_score
                fid = fid_score.calculate_fid_given_paths(
                    [fid_hr_dir, fid_sr_dir], batch_size=50,
                    device=DEVICE, dims=2048)
                for r in all_results:
                    if r["model"] == model_name and r["dataset"] == "BSD100":
                        r["fid"] = float(fid)
                print(f"  {model_name} FID: {fid:.2f}")
                # Save immediately so FID is persisted if the next model crashes
                _save_results_csv(all_results, config)
                _save_results_latex(all_results, config)
            except Exception as e:
                print(f"  ⚠️  FID computation failed for {model_name}: {e}")

    # ── Final save ──
    _save_results_csv(all_results, config)
    _save_results_latex(all_results, config)
    return all_results


def _save_results_csv(results, config):
    """Save evaluation results as CSV."""
    path = os.path.join(config["results_dir"], "metrics_summary.csv")
    keys = ["model", "dataset", "scale", "psnr", "ssim", "lpips", "fid", "time_avg"]
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader()
        w.writerows(results)
    print(f">CSV saved: {path}")


def _save_results_latex(results, config):
    """Save evaluation results as LaTeX table."""
    path = os.path.join(config["results_dir"], "metrics_table.tex")
    with open(path, "w") as f:
        f.write("\\begin{table}[h]\n\\centering\n")
        f.write(f"\\caption{{Super-resolution results at $\\times${config['scale_factor']}}}\n")
        f.write("\\begin{tabular}{llcccc}\n\\toprule\n")
        f.write("Model & Dataset & PSNR$\\uparrow$ & SSIM$\\uparrow$ "
                "& LPIPS$\\downarrow$ & FID$\\downarrow$ \\\\\n\\midrule\n")
        for r in results:
            fid_str = f"{r['fid']:.2f}" if isinstance(r.get("fid"), float) else "—"
            f.write(f"{r['model']} & {r['dataset']} & {r['psnr']:.2f} & "
                    f"{r['ssim']:.4f} & {r['lpips']:.4f} & {fid_str} \\\\\n")
        f.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")
    print(f">LaTeX table saved: {path}")


print(">Evaluation functions defined")

>Evaluation functions defined


## Section 12 — Ablation Study
For the DDPM model, ablate over: (a) diffusion steps, (b) noise schedule, (c) scale.

In [16]:
@torch.no_grad()
def run_ablation(config):
    """Run ablation experiments on the trained DDPM model.

    Tests different diffusion step counts and noise schedules on Set14.
    Saves results as CSV and bar charts.
    """
    print("=" * 60)
    print("  🔬 ABLATION STUDY")
    print("=" * 60)

    # Load DDPM model
    unet = UNet(
        in_ch=6, out_ch=3, base_ch=config["unet_base_ch"],
        ch_mults=config["unet_ch_mults"], num_res=config["unet_num_res"],
        attn_res=config["unet_attn_res"], dropout=config["unet_dropout"],
    ).to(DEVICE)
    best_path = os.path.join(config["checkpoint_dir"], "ddpm", "best_model.pt")
    if not os.path.exists(best_path):
        print("⚠️  No DDPM best_model.pt found. Skipping ablation.")
        return []
    unet.load_state_dict(torch.load(best_path, map_location=DEVICE,
                                    weights_only=False))
    unet.eval()

    lpips_fn = lpips.LPIPS(net="alex").to(DEVICE)
    # Use Set14 for ablation
    test_ds = SRTestDataset(config["set14_dir"], scale_factor=config["scale_factor"])
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

    ablation_results = []

    # (a) Ablate over diffusion steps
    print("\n── (a) Diffusion Steps Ablation ──")
    for steps in config["ablation_steps"]:
        schedule = make_diffusion_schedule(
            config["noise_schedule"], steps,
            config["beta_start"], config["beta_end"])
        ddim = DDIMSampler(schedule, min(steps, config["ddim_sampling_steps"]))
        psnrs, lps = [], []
        for batch in tqdm(test_loader, desc=f"  Steps={steps}"):
            hr  = batch["hr"].to(DEVICE)
            bic = batch["bicubic"].to(DEVICE)
            sr  = ddim.sample(unet, bic)
            sr_np, hr_np = tensor_to_np(sr[0]), tensor_to_np(hr[0])
            p, _ = calc_psnr_ssim(sr_np, hr_np, border=config["eval_border_crop"])
            psnrs.append(p)
            lps.append(lpips_fn(sr * 0.5 + 0.5, hr * 0.5 + 0.5).mean().item())
        ablation_results.append({
            "ablation": "steps", "value": steps,
            "psnr": np.mean(psnrs), "lpips": np.mean(lps)})
        print(f"    Steps={steps}: PSNR={np.mean(psnrs):.2f} LPIPS={np.mean(lps):.4f}")

    # (b) Ablate over noise schedule
    print("\n── (b) Noise Schedule Ablation ──")
    for sched_type in config["ablation_schedules"]:
        schedule = make_diffusion_schedule(
            sched_type, config["diffusion_steps"],
            config["beta_start"], config["beta_end"])
        ddim = DDIMSampler(schedule, config["ddim_sampling_steps"])
        psnrs, lps = [], []
        for batch in tqdm(test_loader, desc=f"  Schedule={sched_type}"):
            hr  = batch["hr"].to(DEVICE)
            bic = batch["bicubic"].to(DEVICE)
            sr  = ddim.sample(unet, bic)
            sr_np, hr_np = tensor_to_np(sr[0]), tensor_to_np(hr[0])
            p, _ = calc_psnr_ssim(sr_np, hr_np, border=config["eval_border_crop"])
            psnrs.append(p)
            lps.append(lpips_fn(sr * 0.5 + 0.5, hr * 0.5 + 0.5).mean().item())
        ablation_results.append({
            "ablation": "schedule", "value": sched_type,
            "psnr": np.mean(psnrs), "lpips": np.mean(lps)})
        print(f"    {sched_type}: PSNR={np.mean(psnrs):.2f} LPIPS={np.mean(lps):.4f}")

    # Save ablation results
    abl_path = os.path.join(config["results_dir"], "ablation_results.csv")
    with open(abl_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["ablation", "value", "psnr", "lpips"])
        w.writeheader()
        w.writerows(ablation_results)

    # Plot
    _plot_ablation(ablation_results, config)
    print(f">Ablation results saved to {abl_path}")
    return ablation_results


def _plot_ablation(results, config):
    """Generate bar charts for ablation results."""
    for abl_type in ["steps", "schedule"]:
        subset = [r for r in results if r["ablation"] == abl_type]
        if not subset:
            continue
        labels = [str(r["value"]) for r in subset]
        psnrs  = [r["psnr"] for r in subset]
        lps    = [r["lpips"] for r in subset]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        ax1.bar(labels, psnrs, color="#4A90D9")
        ax1.set_title(f"PSNR vs {abl_type}")
        ax1.set_ylabel("PSNR (dB)")
        ax2.bar(labels, lps, color="#E74C3C")
        ax2.set_title(f"LPIPS vs {abl_type}")
        ax2.set_ylabel("LPIPS")
        plt.tight_layout()
        plt.savefig(os.path.join(config["visual_dir"], f"ablation_{abl_type}.png"),
                    dpi=150)
        plt.close()


print(">Ablation functions defined")

>Ablation functions defined


## Section 13 — Visualisation
Side-by-side comparison grids: LR | Bicubic | ESRGAN | DDPM | DDIM | HR

In [17]:
@torch.no_grad()
def create_comparison_visuals(config, max_images=5):
    """Create side-by-side comparison images for all models on each test set.

    Saves to /kaggle/working/visuals/
    """
    print("=" * 60)
    print("  🖼  GENERATING COMPARISON VISUALS")
    print("=" * 60)

    _, test_loaders = build_dataloaders(config)
    schedule = make_diffusion_schedule(
        config["noise_schedule"], config["diffusion_steps"],
        config["beta_start"], config["beta_end"])
    ddim_sampler = DDIMSampler(schedule, config["ddim_sampling_steps"])

    # Load models
    unet, esrgan = None, None
    unet_path = os.path.join(config["checkpoint_dir"], "ddpm", "best_model.pt")
    if os.path.exists(unet_path):
        unet = UNet(in_ch=6, out_ch=3, base_ch=config["unet_base_ch"],
                     ch_mults=config["unet_ch_mults"],
                     num_res=config["unet_num_res"],
                     attn_res=config["unet_attn_res"]).to(DEVICE)
        unet.load_state_dict(torch.load(unet_path, map_location=DEVICE,
                                        weights_only=False))
        unet.eval()

    esrgan_path = os.path.join(config["checkpoint_dir"], "esrgan", "best_model.pt")
    if os.path.exists(esrgan_path):
        esrgan = RRDBNet(in_nc=3, out_nc=3, nf=64,
                          nb=config["esrgan_num_rrdb"],
                          scale=config["scale_factor"]).to(DEVICE)
        esrgan.load_state_dict(torch.load(esrgan_path, map_location=DEVICE,
                                          weights_only=False))
        esrgan.eval()

    for ds_name, loader in test_loaders.items():
        for idx, batch in enumerate(loader):
            if idx >= max_images:
                break
            hr  = batch["hr"].to(DEVICE)
            lr  = batch["lr"].to(DEVICE)
            bic = batch["bicubic"].to(DEVICE)
            fname = batch.get("filename", [f"{idx:03d}.png"])[0]

            panels = [
                ("LR (↑bicubic)", tensor_to_np(bic[0])),
                ("Bicubic", tensor_to_np(bic[0])),
            ]
            if esrgan is not None:
                sr_gan = esrgan(lr)
                panels.append(("ESRGAN", tensor_to_np(sr_gan[0])))
            if unet is not None:
                sr_ddpm = ddpm_sample(unet, bic, schedule)
                panels.append(("DDPM", tensor_to_np(sr_ddpm[0])))
                sr_ddim = ddim_sampler.sample(unet, bic)
                panels.append(("DDIM", tensor_to_np(sr_ddim[0])))
            panels.append(("Ground Truth", tensor_to_np(hr[0])))

            # Plot
            n = len(panels)
            fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
            for ax, (title, img) in zip(axes, panels):
                ax.imshow(img)
                ax.set_title(title, fontsize=10)
                ax.axis("off")
            plt.suptitle(f"{ds_name} — {fname} (×{config['scale_factor']})",
                         fontsize=12)
            plt.tight_layout()
            save_path = os.path.join(
                config["visual_dir"],
                f"comparison_{ds_name}_x{config['scale_factor']}_{fname}")
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"  Saved: {save_path}")

    print(">Comparison visuals saved")


print(">Visualisation functions defined")

>Visualisation functions defined


## Section 14 — Results Summary & Loss Curves

In [18]:
def plot_loss_curves(config):
    """Plot training loss curves for DDPM and ESRGAN from checkpoint histories."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, name in zip(axes, ["ddpm", "esrgan"]):
        ckpt_path = os.path.join(config["checkpoint_dir"], name, "epoch_latest.pt")
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            hist = ckpt.get("loss_history", [])
            if hist:
                ax.plot(hist, linewidth=1.5)
                ax.set_xlabel("Epoch")
                ax.set_ylabel("Loss")
        ax.set_title(f"{name.upper()} Training Loss")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(config["visual_dir"], "loss_curves.png"), dpi=150)
    plt.close()
    print(">Loss curves saved")


print(">Results summary functions defined")

>Results summary functions defined


## Section 15 — Checkpoint Persistence (Kaggle Dataset Push)
Use this section to push checkpoints to a Kaggle Dataset so they survive
beyond the session's output expiry.

In [19]:
import os
import json
def kaggle_whoami():
    """Return the Kaggle username from the configured API token."""
    import subprocess
    r = subprocess.run(["kaggle", "config", "view"], capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if "username" in line.lower():
            return line.split(":")[-1].strip()
    raise RuntimeError("Could not detect Kaggle username. Check API token.")


def push_checkpoints_to_kaggle(dataset_slug, model_name):
    """Push a specific model checkpoint folder to a Kaggle dataset.

    Args:
        dataset_slug: name for the dataset (e.g. "ddpm-sr-checkpoints")
        model_name:   subfolder to push — "ddpm" or "esrgan"

    Usage:
        push_checkpoints_to_kaggle("ddpm-sr-checkpoints", "ddpm")
        push_checkpoints_to_kaggle("esrgan-sr-checkpoints", "esrgan")
    """
    import subprocess, shutil, tempfile

    username = kaggle_whoami()
    src_dir = f"/kaggle/working/checkpoints/{model_name}"

    if not os.path.exists(src_dir):
        print(f"❌ Source dir not found: {src_dir}")
        return

    tmp_dir = tempfile.mkdtemp(prefix="kaggle_push_")
    try:
        dst = os.path.join(tmp_dir, model_name)
        shutil.copytree(src_dir, dst)

        meta = {
            "title": dataset_slug,
            "id": f"{username}/{dataset_slug}",
            "licenses": [{"name": "CC0-1.0"}]
        }
        with open(os.path.join(tmp_dir, "dataset-metadata.json"), "w") as f:
            json.dump(meta, f, indent=2)

        print(f">Pushing {src_dir} → {username}/{dataset_slug}")
        for root, _, files in os.walk(dst):
            for fname in files:
                fpath = os.path.join(root, fname)
                print(f"  {os.path.relpath(fpath, tmp_dir)}  ({os.path.getsize(fpath)/1e6:.1f} MB)")

        result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", tmp_dir],
            capture_output=True, text=True)

        if "already exists" in result.stderr.lower() or result.returncode != 0:
            subprocess.run(
                ["kaggle", "datasets", "version", "-p", tmp_dir,
                 "-m", f"epoch {time.strftime('%Y%m%d_%H%M')}"],
                check=True)
            print(">Dataset version updated ✅")
        else:
            print(">Dataset created ✅")

        print(f">https://www.kaggle.com/datasets/{username}/{dataset_slug}")
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

## Section 16 — RUN CELLS
Run all three cells sequentially in a single session.
Each saves checkpoints on completion — if the session dies, re-running resumes automatically.


In [20]:
# ──────────── 1. Train DDPM ────────────
ddpm_model, ddpm_schedule = train_ddpm(CONFIG)


  🔵 DDPM / SR3 TRAINING
>Batch preview saved to visuals/batch_preview.png
U-Net params: 57,387,267
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 210MB/s]


>Resumed ddpm from epoch 100 (best PSNR=19.22 dB)
>DDPM training complete


In [21]:
# ──────────── 2. Train ESRGAN ────────────

esrgan_model = train_esrgan(CONFIG)


  🟢 ESRGAN TRAINING
⚠️  Loaded pretrained ESRGAN (strict=False) from /kaggle/input/datasets/ahmedalizahid/esrgan-pretrained/RRDB_PSNR_x4.pth
Generator params: 16,919,555
Discriminator params: 7,173,897
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 201MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
>Resumed esrgan from epoch 200 (best PSNR=27.77 dB)
>ESRGAN training complete


In [22]:
# ──────────── 3. Evaluate & Visualise ────────────
results  = evaluate_all_models(CONFIG)
ablation = run_ablation(CONFIG)
create_comparison_visuals(CONFIG)
plot_loss_curves(CONFIG)


  📊 FULL EVALUATION
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
>Loaded DDPM best model
>Loaded ESRGAN best model
>Resuming eval — 9 (model, dataset) pairs already done; 0 FID(s) already computed

── Evaluating on Set5 (5 images) ──
  ⏩ Skipping DDPM/Set5 (already in CSV)
  ⏩ Skipping DDIM/Set5 (already in CSV)
  ⏩ Skipping ESRGAN/Set5 (already in CSV)

── Evaluating on Set14 (14 images) ──
  ⏩ Skipping DDPM/Set14 (already in CSV)
  ⏩ Skipping DDIM/Set14 (already in CSV)
  ⏩ Skipping ESRGAN/Set14 (already in CSV)

── Evaluating on BSD100 (100 images) ──
  ⏩ Skipping DDPM/BSD100 (already in CSV)
  ⏩ Skipping DDIM/BSD100 (already in CSV)
  ⏩ Skipping ESRGAN/BSD100 (already in CSV)

── Computing FID on BSD100 ──
  ⚠️  Resuming DDPM FID image generation (35/100 already done, skipping those)...


  DDPM FID images:   0%|          | 0/100 [00:00<?, ?it/s]

Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth


100%|██████████| 91.2M/91.2M [00:00<00:00, 191MB/s]
  0%|          | 0/2 [00:00<?, ?it/s]


  ⚠️  FID computation failed for DDPM: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/collate.py", line 401, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/collate.py", line 155, in collate
    return collate_fn_map[elem_type](batch, collate_fn_map=collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

...

    Steps=300: PSNR=13.11 LPIPS=0.9681

── (b) Noise Schedule Ablation ──


  Schedule=linear:   0%|          | 0/14 [00:00<?, ?it/s]

    linear: PSNR=12.90 LPIPS=0.9525


  Schedule=cosine:   0%|          | 0/14 [00:00<?, ?it/s]

    cosine: PSNR=12.78 LPIPS=1.0068
>Ablation results saved to /kaggle/working/results/ablation_results.csv
  🖼  GENERATING COMPARISON VISUALS
  Saved: /kaggle/working/visuals/comparison_Set5_x4_baby.png
  Saved: /kaggle/working/visuals/comparison_Set5_x4_bird.png
  Saved: /kaggle/working/visuals/comparison_Set5_x4_butterfly.png
  Saved: /kaggle/working/visuals/comparison_Set5_x4_head.png
  Saved: /kaggle/working/visuals/comparison_Set5_x4_woman.png
  Saved: /kaggle/working/visuals/comparison_Set14_x4_baboon.png
  Saved: /kaggle/working/visuals/comparison_Set14_x4_barbara.png
  Saved: /kaggle/working/visuals/comparison_Set14_x4_bridge.png
  Saved: /kaggle/working/visuals/comparison_Set14_x4_coastguard.png
  Saved: /kaggle/working/visuals/comparison_Set14_x4_comic.png
  Saved: /kaggle/working/visuals/comparison_BSD100_x4_101085.png
  Saved: /kaggle/working/visuals/comparison_BSD100_x4_101087.png
  Saved: /kaggle/working/visuals/comparison_BSD100_x4_102061.png
  Saved: /kaggle/working/vi

In [23]:
# ──────────── Optional: persist checkpoints to Kaggle dataset ────────────
# 
#push_checkpoints_to_kaggle("ddpm-sr-checkpoints", "ddpm")
#push_checkpoints_to_kaggle("esrgan-sr-checkpoints", "esrgan")
